## Contribution

Task 1: SIVAKUMAR MADHUMITHA

Task 2: TISHA LEAW WAN NING

Task 3: JOHANNAS BIN JUMARI

Task 4: all

### Step 1: Install necesscary packages

In [1]:
%cd /content
!git clone https://github.com/efilyssem/SC3000-nanogpt.git
%cd SC3000-nanogpt/dpo

!gdown 1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ -O ../sft/gpt.pt
!ls -lh ../sft

/content
fatal: destination path 'SC3000-nanogpt' already exists and is not an empty directory.
/content/SC3000-nanogpt/dpo
Downloading...
From (original): https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ
From (redirected): https://drive.google.com/uc?id=1gIZw-HAB-tHtEYCmNugwlIV7R3WsgjjZ&confirm=t&uuid=77e90b61-15f2-4b4f-a1d6-33538b3fe443
To: /content/SC3000-nanogpt/sft/gpt.pt
100% 106M/106M [00:00<00:00, 156MB/s] 
total 102M
-rw-r--r-- 1 root root 102M Aug 28  2025 gpt.pt
-rw-r--r-- 1 root root  795 Nov  4 06:45 meta.pkl


In [2]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In our project, we have chosen to use 20 epchos as it provides better level of accuracy when measuring the loss function values. Though increasing the epchos will not automatically increase the accuracy level, it will provide more opportunities to learn and refine its parameters. It would also help to achieve better convergence by repeatedly learning from the training dataset.
We have also changed the top_k value from 200 to 1. (used in the testing segment) This will reduce the amount of randomness and produces more deterministic outputs when evaluating the model's performance because we are now consistently selecting the most probable token at each step instead of randomly sampling from the 200 most probable tokens. 

Together we are now able to ensure increased amount of determinance in the results being generated along with better learning measures of the parameter

In [3]:
import sys
import os
sys.path.append(os.path.abspath(".."))
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 20
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 1
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [4]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [5]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

GPT(
  (transformer): ModuleDict(
    (wte): Embedding(74, 348)
    (wpe): Embedding(256, 348)
    (drop): Dropout(p=0.2, inplace=False)
    (h): ModuleList(
      (0-5): 6 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_features=348, out_features=1044, bias=False)
          (c_proj): Linear(in_features=348, out_features=348, bias=False)
          (attn_dropout): Dropout(p=0.2, inplace=False)
          (resid_dropout): Dropout(p=0.2, inplace=False)
        )
        (ln_2): LayerNorm()
        (mlp): MLP(
          (c_fc): Linear(in_features=348, out_features=1392, bias=False)
          (gelu): GELU(approximate='none')
          (c_proj): Linear(in_features=1392, out_features=348, bias=False)
          (dropout): Dropout(p=0.2, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm()
  )
  (lm_head): Linear(in_features=348, out_features=74, bias=False)
)

### Step 5: Load Data (**students are required to complete this part!**)

In Step 5, we first generated approximately 33,000 unique positive-negative response pairs using arithmetic and algebraic manipulation. Each pair consists of a positive and a negative response. We validated the mathematical correctness of the answers and explanations before repeating the unique pairs three times to reach approximately over 100,000 pairs for model training. We then shuffled the dataset to randomise the order of the pairs.

In [6]:
import json
import random
import re
from collections import Counter
from pathlib import Path

# CONFIGURATION
MIN_VALUE = 0
MAX_VALUE = 99          # applies to every number in the question, including results
REPEAT = 3              # to generate total 100k pairs
SHUFFLE_AFTER_REPEAT = True # so that copies are not in the same exact order

REMOVE_TRIVIAL = False
TARGET_SIZE = None
SEED = 42

OUTPUT = Path("pos_neg_pairs_all.json")

NEG_TEXT = "Sorry, I do not know."

skipped = []


# STEP 1: Calculate the correct answer independently
def calculate_answer(question):

    q = question.replace(" ", "")

    # Basic arithmetic: a op b = ?
    m = re.fullmatch(r"(\d+)([+*/-])(\d+)=\?", q)

    if m:
        a, op, b = m.groups()
        a, b = int(a), int(b)

        if op == "+":
            return a + b
        if op == "-":
            return a - b
        if op == "*":
            return a * b

        if b == 0 or a % b != 0:
            raise ValueError("Invalid division")

        return a // b

    # Algebra: x on the left, x op a = result
    m = re.fullmatch(r"x([+*/-])(\d+)=(-?\d+),x=\?", q)

    if m:
        op, a, result = m.groups()
        a, result = int(a), int(result)

        if op == "+":
            return result - a

        if op == "-":
            return result + a

        if op == "*":
            if a == 0:
                raise ValueError("Infinite solutions")

            if result % a != 0:
                raise ValueError("Non-integer solution")

            return result // a

        if a == 0:
            raise ValueError("Division by zero")

        return result * a

    # Algebra: x on the right, a op x = result
    m = re.fullmatch(r"(\d+)([+*/-])x=(-?\d+),x=\?", q)

    if m:
        a, op, result = m.groups()
        a, result = int(a), int(result)

        if op == "+":
            return result - a

        if op == "-":
            return a - result

        if op == "*":
            if a == 0:
                raise ValueError("Infinite solutions")

            if result % a != 0:
                raise ValueError("Non-integer solution")

            return result // a

        if a == 0 or result == 0:
            raise ValueError("No unique solution")

        if a % result != 0:
            raise ValueError("Non-integer solution")

        return a // result

    raise ValueError("Invalid question format")

  # STEP 1b: Build the explanation for the positive answer
def explain(question, answer):

    q = question.replace(" ", "")

    # Basic arithmetic: "26+760 equals 786"
    m = re.fullmatch(r"(\d+)([+*/-])(\d+)=\?", q)

    if m:
        a, op, b = m.groups()
        return f"{a}{op}{b} equals {answer}"

    # x on the left: x op a = result -> apply the inverse operation to result
    m = re.fullmatch(r"x([+*/-])(\d+)=(-?\d+),x=\?", q)

    if m:
        op, a, result = m.groups()
        inverse = {"+": "-", "-": "+", "*": "/", "/": "*"}[op]
        return f"{result}{inverse}{a} equals {answer}"

    # x on the right: a op x = result
    m = re.fullmatch(r"(\d+)([+*/-])x=(-?\d+),x=\?", q)

    if m:
        a, op, result = m.groups()

        if op == "+":
            return f"{result}-{a} equals {answer}"
        if op == "-":
            return f"{a}-{result} equals {answer}"
        if op == "*":
            return f"{result}/{a} equals {answer}"
        return f"{a}/{result} equals {answer}"

    raise ValueError("Invalid question format")


# STEP 2: Add a valid question (invalid ones are skipped, not raised)
def add_pair(dataset, seen, question, answer):

    # Avoid duplicate questions
    if question in seen:
        return

    # Exclude anything invalid instead of raising
    try:
        calculated = calculate_answer(question)
    except ValueError:
        skipped.append(question)
        return

    if calculated != answer:
        skipped.append(question)
        return

    seen.add(question)

    dataset.append({
        "negative": f"{question} {NEG_TEXT}",
        "positive": f"{question} The answer is {answer} because {explain(question, answer)}."
    })


# STEP 3: Generate all combinations
# Each question type has its own loop, so a filter in one type
# can never skip questions in another type.
def generate_dataset():

    dataset = []
    seen = set()

    numbers = range(MIN_VALUE, MAX_VALUE + 1)

    # Smallest allowed operand for + and - (0 is trivial)
    lo = max(MIN_VALUE, 1) if REMOVE_TRIVIAL else MIN_VALUE

    # Smallest allowed operand for * in basic arithmetic (0 and 1 are trivial)
    mul_lo = 2 if REMOVE_TRIVIAL else 0

    # Smallest allowed divisor / multiplier in algebra (0 never valid, 1 trivial)
    div_lo = 2 if REMOVE_TRIVIAL else 1

    # Algebra operands and answers
    alg = range(lo, MAX_VALUE + 1)

    # ---------------------------------------------
    # TYPE 1: Basic arithmetic, a op b = ?
    # ---------------------------------------------
    for a in numbers:
        for b in numbers:

            # a + b (result stays within range)
            if a >= lo and b >= lo and a + b <= MAX_VALUE:
                add_pair(dataset, seen, f"{a}+{b}=?", a + b)

            # a - b (no negative answers)
            if b >= lo and a >= b:
                add_pair(dataset, seen, f"{a}-{b}=?", a - b)

            # a * b (result stays within range)
            if a >= mul_lo and b >= mul_lo and a * b <= MAX_VALUE:
                add_pair(dataset, seen, f"{a}*{b}=?", a * b)

            # a / b (no /0, no fractional answers)
            if b >= div_lo and a >= lo and a % b == 0:
                add_pair(dataset, seen, f"{a}/{b}=?", a // b)

    # ---------------------------------------------
    # TYPE 2: Unknown on the left, x op a = result
    # x is the answer
    # ---------------------------------------------
    for a in alg:
        for x in alg:

            if x + a <= MAX_VALUE:
                add_pair(dataset, seen, f"x+{a}={x + a},x=?", x)

            if x >= a:
                add_pair(dataset, seen, f"x-{a}={x - a},x=?", x)

            if a >= div_lo and x * a <= MAX_VALUE:
                add_pair(dataset, seen, f"x*{a}={x * a},x=?", x)

            if a >= div_lo and x % a == 0:
                add_pair(dataset, seen, f"x/{a}={x // a},x=?", x)

    # ---------------------------------------------
    # TYPE 3: Unknown on the right, a op x = result
    # x is the answer
    # ---------------------------------------------
    for a in alg:
        for x in alg:

            if a + x <= MAX_VALUE:
                add_pair(dataset, seen, f"{a}+x={a + x},x=?", x)

            if a >= x:
                add_pair(dataset, seen, f"{a}-x={a - x},x=?", x)

            if a >= div_lo and a * x <= MAX_VALUE:
                add_pair(dataset, seen, f"{a}*x={a * x},x=?", x)

            # 0 / x = 0 has infinite solutions; x = 1 is trivial when filtering
            if x >= div_lo and a > 0 and a % x == 0:
                add_pair(dataset, seen, f"{a}/x={a // x},x=?", x)

    return dataset


# STEP 4: Optional random sample down to a target size
def sample_dataset(dataset):

    if TARGET_SIZE is None or TARGET_SIZE >= len(dataset):
        return dataset

    rng = random.Random(SEED)

    return rng.sample(dataset, TARGET_SIZE)


# STEP 5: Count pairs per question type
def question_type(question):

    if question.endswith(",x=?"):
        return "x_left" if question.startswith("x") else "x_right"

    return "basic"


def count_types(dataset):

    counts = Counter()

    for pair in dataset:
        question = pair["positive"].split(" The answer is ")[0]
        counts[question_type(question)] += 1

    return counts


# STEP 6: Validate the dataset
def validate_dataset(dataset):

    seen = set()
    suffix = f" {NEG_TEXT}"

    for pair in dataset:

        negative = pair["negative"]
        positive = pair["positive"]

        assert negative.endswith(suffix)
        question = negative[:-len(suffix)]

        # Check duplicates
        assert question not in seen
        seen.add(question)

        # Check positive format
        prefix = f"{question} The answer is "

        assert positive.startswith(prefix)
        assert positive.endswith(".")

        m = re.fullmatch(r"(-?\d+) because (.+)\.", positive[len(prefix):])
        assert m, f"Bad positive format: {positive}"

        answer = int(m.group(1))
        explanation = m.group(2)

        # Check mathematical correctness of the answer
        assert calculate_answer(question) == answer

        # Check the explanation matches the expected one
        assert explanation == explain(question, answer)

        # Check the explanation's own arithmetic is true
        lhs, rhs = explanation.split(" equals ")
        assert eval(lhs) == int(rhs)   # lhs only contains digits and + - * /

    assert len(dataset) == len(seen)

    print("All validation checks passed!")

# STEP 7: Save dataset
def main():

    dataset = sample_dataset(generate_dataset())

    # Validate and count the unique pairs BEFORE repeating
    validate_dataset(dataset)
    counts = count_types(dataset)
    unique_total = len(dataset)

    # Repeat the dataset
    dataset = dataset * REPEAT

    if SHUFFLE_AFTER_REPEAT:
        random.Random(SEED).shuffle(dataset)

    OUTPUT.parent.mkdir(parents=True, exist_ok=True)

    with OUTPUT.open("w", encoding="utf-8") as f:
        json.dump(dataset, f, ensure_ascii=False, indent=2)

    print("\n========== DATASET SUMMARY ==========")

    print(f"Number range: {MIN_VALUE} - {MAX_VALUE}")
    print("Operators: +, -, *, /")
    print(f"Remove trivial pairs: {REMOVE_TRIVIAL}")

    for name in ("basic", "x_left", "x_right"):
        print(f"  {name:<8} {counts[name]:>6,}  (x{REPEAT} in file)")

    print(f"\nUnique pairs: {unique_total:,}")
    print(f"Repeat factor: {REPEAT}")
    print(f"Total pairs in file: {len(dataset):,}")
    print(f"Skipped invalid pairs: {len(skipped)}")
    print(f"Dataset saved to: {OUTPUT}")

    print("\nFirst 5 examples:")

    for pair in dataset[:5]:
        print(json.dumps(pair, indent=2))


if __name__ == "__main__":
    main()

All validation checks passed!

========== DATASET SUMMARY ==========
Number range: 0 - 99
Operators: +, -, *, /
Remove trivial pairs: False
  basic    11,344  (x3 in file)
  x_left   11,244  (x3 in file)
  x_right  11,145  (x3 in file)

Unique pairs: 33,733
Repeat factor: 3
Total pairs in file: 101,199
Skipped invalid pairs: 0
Dataset saved to: pos_neg_pairs_all.json

First 5 examples:
{
  "negative": "62+6=? Sorry, I do not know.",
  "positive": "62+6=? The answer is 68 because 62+6 equals 68."
}
{
  "negative": "43/x=43,x=? Sorry, I do not know.",
  "positive": "43/x=43,x=? The answer is 1 because 43/43 equals 1."
}
{
  "negative": "x-1=87,x=? Sorry, I do not know.",
  "positive": "x-1=87,x=? The answer is 88 because 87+1 equals 88."
}
{
  "negative": "58*x=0,x=? Sorry, I do not know.",
  "positive": "58*x=0,x=? The answer is 0 because 0/58 equals 0."
}
{
  "negative": "x-41=2,x=? Sorry, I do not know.",
  "positive": "x-41=2,x=? The answer is 43 because 2+41 equals 43."
}


In [7]:
data_path = Path("pos_neg_pairs_all.json")
if not data_path.exists():
    data_path = Path("dpo/pos_neg_pairs_all.json")

with open(data_path, "r", encoding="utf-8") as f:
    loaded_pairs = json.load(f)
print("Number of pairs:", len(loaded_pairs))

Number of pairs: 101199


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

This is where we implement the AdamW optimizer, which determines how the model's parameters should be updated to minimise the loss function, and the learning rate scheduler, which controls how large the parameter updates should be throughout training. This helps the model train more stably.

We have also chosen to employ `torch.optim.lr_scheduler.LambdaLR` because, instead of maintaining a constant learning rate, we are now able to control how the learning rate changes throughout the training process in Step 7. It combines linear warmup with cosine decay, where the first 40% of the training steps are used for warmup, during which the learning rate gradually increases from 0 to `base_lr`.

Once the warmup is over, the learning rate gradually decreases using cosine decay, eventually reaching 5% of the base learning rate instead of zero. As such, this provides greater flexibility in controlling how the learning rate changes, making it better suited to our training configuration.

In [8]:
import math
# recommend to use the AdamW optimizer
optimizer = torch.optim.AdamW(
    gpt.parameters(),
    lr=base_lr,
    betas=(0.9, 0.95),
    eps=1e-8,
    weight_decay=0.1
)

total_steps = epochs * (len(loaded_pairs)//batch_size)
warmup_ratio = 0.4
warmup_steps = max(1, int(warmup_ratio * total_steps))
min_lr_ratio = 0.05
min_lr = base_lr * min_lr_ratio
def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps

    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))

    return (min_lr / base_lr) + (1.0 - min_lr / base_lr) * cosine

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda
)

### Step 7: Begin training (**students are required to complete this part!**)

In this stage, we trained our pretrained GPT model using the preference dataset of slightly over 100k chosen–rejected response pairs prepared in Step 5.

Using the AdamW optimizer and LambdaLR learning rate scheduler configured in Step 6, we trained the model for 20 epochs. During each epoch, the dataset was processed in batches of 64, where the DPO loss was calculated to measure how well the model distinguished between preferred and rejected responses. Through backpropagation and parameter updates, the model was progressively fine-tuned to favour the preferred responses, as we see the DPO value decreasing over time after every round of iteration. 

Throughout training, we monitored the loss function to evaluate the model's learning progress and convergence. The resulting trained model was then used in Step 8 to evaluate its performance on test cases.

In [9]:

lines = loaded_pairs.copy()
gpt.train()
total_steps = len(lines) // batch_size
for epoch in range(epochs):
    pbar = tqdm(get_batches(lines, batch_size))
    for step, (neg_tensor,pos_tensor) in enumerate(pbar):
        ###########################################################
        # Please complete the training code here!
        # Examples:
        # ...
        # neg_logprob
        # pos_logprob
        # loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1
        # ...
        ###########################################################
        neg_logprob = compute_logprob(neg_tensor)
        pos_logprob = compute_logprob(pos_tensor)

        loss = -F.logsigmoid((pos_logprob - neg_logprob) / beta).mean() - pos_logprob.mean() * 0.1

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        pbar.set_postfix(loss=loss.item())

    ckpt_path = f"./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt['model_args'],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

1581it [04:17,  6.14it/s, loss=0.0541]


Saved checkpoint to ./dpo.pt


1581it [04:17,  6.15it/s, loss=0.0299]


Saved checkpoint to ./dpo.pt


1581it [04:17,  6.15it/s, loss=0.0236]


Saved checkpoint to ./dpo.pt


... (30 outputs left out)

1581it [04:16,  6.16it/s, loss=0.0169]


Saved checkpoint to ./dpo.pt


1581it [04:16,  6.16it/s, loss=0.0172]


Saved checkpoint to ./dpo.pt


### Step 8: Begin testing (**students are required to complete this part!**)

After training, we loaded the fine tuneed model and tested it using arithmetic and algebric questions. We generated responses to evaluate whether the model could produce correct mathematical responses. We also had 100 test cases to ensure the accuracy of the model 

In [10]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set:
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # ...
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        # ...
        ###########################################################
        x = torch.tensor(prompt_ids, dtype=torch.long).unsqueeze(0).to(device)

        y, hidden_state = gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)

        result = decode(y[0].tolist())
        print(result)

17+19=? The answer is 36 because 17+19 equals 36.
3*17=? The answer is 51 because 3*17 equals 51.
72/4=? The answer is 18 because 72/4 equals 18.
72-x=34,x=? The answer is 38 because 72-34 equals 38.
x*11=44,x=? The answer is 4 because 44/11 equals 4.
3*17=? The answer is 51 because 3*17 equals 51.
72/4=? The answer is 18 because 72/4 equals 18.
72-x=34,x=? The answer is 38 because 72-34 equals 38.


## Additional Test Case

In [11]:
import json
import random
import re
from pathlib import Path

MIN_VALUE = 0
MAX_VALUE = 99          # applies to every number in the question, including results
REMOVE_TRIVIAL = False
SEED = 42
NUM_TESTS = 100
OUTPUT = Path("test_cases_100.json")


# Independent solver (same as the pair generator)
def calculate_answer(question):
    q = question.replace(" ", "")

    m = re.fullmatch(r"(\d+)([+*/-])(\d+)=\?", q)
    if m:
        a, op, b = m.groups()
        a, b = int(a), int(b)
        if op == "+": return a + b
        if op == "-": return a - b
        if op == "*": return a * b
        if b == 0 or a % b != 0:
            raise ValueError("Invalid division")
        return a // b

    m = re.fullmatch(r"x([+*/-])(\d+)=(-?\d+),x=\?", q)
    if m:
        op, a, result = m.groups()
        a, result = int(a), int(result)
        if op == "+": return result - a
        if op == "-": return result + a
        if op == "*":
            if a == 0: raise ValueError("Infinite solutions")
            if result % a != 0: raise ValueError("Non-integer solution")
            return result // a
        if a == 0: raise ValueError("Division by zero")
        return result * a

    m = re.fullmatch(r"(\d+)([+*/-])x=(-?\d+),x=\?", q)
    if m:
        a, op, result = m.groups()
        a, result = int(a), int(result)
        if op == "+": return result - a
        if op == "-": return a - result
        if op == "*":
            if a == 0: raise ValueError("Infinite solutions")
            if result % a != 0: raise ValueError("Non-integer solution")
            return result // a
        if a == 0 or result == 0: raise ValueError("No unique solution")
        if a % result != 0: raise ValueError("Non-integer solution")
        return a // result

    raise ValueError("Invalid question format")


def question_type(question):
    if question.endswith(",x=?"):
        return "x_left" if question.startswith("x") else "x_right"
    return "basic"


# Generate every valid question, grouped by type
def generate_pool():
    pool = {"basic": [], "x_left": [], "x_right": []}
    seen = set()

    numbers = range(MIN_VALUE, MAX_VALUE + 1)
    lo = max(MIN_VALUE, 1) if REMOVE_TRIVIAL else MIN_VALUE
    mul_lo = 2 if REMOVE_TRIVIAL else 0
    div_lo = 2 if REMOVE_TRIVIAL else 1
    alg = range(lo, MAX_VALUE + 1)

    def add(question, answer):
        if question in seen:
            return
        try:
            assert calculate_answer(question) == answer
        except (ValueError, AssertionError):
            return
        seen.add(question)
        pool[question_type(question)].append({"question": question, "answer": answer})

    # TYPE 1: a op b = ?
    for a in numbers:
        for b in numbers:
            if a >= lo and b >= lo and a + b <= MAX_VALUE:
                add(f"{a}+{b}=?", a + b)
            if b >= lo and a >= b:
                add(f"{a}-{b}=?", a - b)
            if a >= mul_lo and b >= mul_lo and a * b <= MAX_VALUE:
                add(f"{a}*{b}=?", a * b)
            if b >= div_lo and a >= lo and a % b == 0:
                add(f"{a}/{b}=?", a // b)

    # TYPE 2: x op a = result, x = ?
    for a in alg:
        for x in alg:
            if x + a <= MAX_VALUE:
                add(f"x+{a}={x + a},x=?", x)
            if x >= a:
                add(f"x-{a}={x - a},x=?", x)
            if a >= div_lo and x * a <= MAX_VALUE:
                add(f"x*{a}={x * a},x=?", x)
            if a >= div_lo and x % a == 0:
                add(f"x/{a}={x // a},x=?", x)

    # TYPE 3: a op x = result, x = ?
    for a in alg:
        for x in alg:
            if a + x <= MAX_VALUE:
                add(f"{a}+x={a + x},x=?", x)
            if a >= x:
                add(f"{a}-x={a - x},x=?", x)
            if a >= div_lo and a * x <= MAX_VALUE:
                add(f"{a}*x={a * x},x=?", x)
            if x >= div_lo and a > 0 and a % x == 0:
                add(f"{a}/x={a // x},x=?", x)

    return pool


# Sample 100, balanced across the three types
def sample_tests(pool, n, seed):
    rng = random.Random(seed)
    types = ["basic", "x_left", "x_right"]
    base, extra = divmod(n, len(types))

    tests = []
    for i, t in enumerate(types):
        k = base + (1 if i < extra else 0)
        for item in rng.sample(pool[t], k):
            tests.append({**item, "type": t})

    rng.shuffle(tests)
    return tests


# Validate every rule on the final test set
def validate(tests):
    seen = set()
    for t in tests:
        q, ans = t["question"], t["answer"]

        assert q not in seen, f"duplicate: {q}"
        seen.add(q)

        assert calculate_answer(q) == ans, f"wrong answer: {q}"

        nums = [int(x) for x in re.findall(r"\d+", q)]
        assert all(MIN_VALUE <= v <= MAX_VALUE for v in nums), f"out of range: {q}"
        assert MIN_VALUE <= ans <= MAX_VALUE, f"answer out of range: {q}"
        assert t["type"] == question_type(q)

    print("All validation checks passed!")


def main():
    pool = generate_pool()
    print({k: len(v) for k, v in pool.items()}, "questions in pool")

    tests = sample_tests(pool, NUM_TESTS, SEED)
    validate(tests)

    with OUTPUT.open("w", encoding="utf-8") as f:
        json.dump(tests, f, ensure_ascii=False, indent=2)

    print(f"Saved {len(tests)} test cases to {OUTPUT}")
    for t in tests[:5]:
        print(t)


main()

{'basic': 11344, 'x_left': 11244, 'x_right': 11145} questions in pool
All validation checks passed!
Saved 100 test cases to test_cases_100.json
{'question': 'x+60=72,x=?', 'answer': 12, 'type': 'x_left'}
{'question': '28+50=?', 'answer': 78, 'type': 'basic'}
{'question': '29-9=?', 'answer': 20, 'type': 'basic'}
{'question': '2*6=?', 'answer': 12, 'type': 'basic'}
{'question': 'x*82=82,x=?', 'answer': 1, 'type': 'x_left'}


In [12]:
from collections import defaultdict
import re

def expected_explanation(question, answer):
    q = question.replace(" ", "")

    m = re.fullmatch(r"(\d+)([+*/-])(\d+)=\?", q)
    if m:
        a, op, b = m.groups()
        return f"{a}{op}{b}={answer}"

    m = re.fullmatch(r"x([+*/-])(\d+)=(-?\d+),x=\?", q)
    if m:
        op, a, result = m.groups()
        inverse = {"+": "-", "-": "+", "*": "/", "/": "*"}[op]
        return f"{result}{inverse}{a}={answer}"

    m = re.fullmatch(r"(\d+)([+*/-])x=(-?\d+),x=\?", q)
    if m:
        a, op, result = m.groups()
        if op == "+": return f"{result}-{a}={answer}"
        if op == "-": return f"{a}-{result}={answer}"
        if op == "*": return f"{result}/{a}={answer}"
        return f"{a}/{result}={answer}"

    raise ValueError(f"Invalid question: {question}")

def parse_output(output, prompt):
    completion = output[len(prompt):] if output.startswith(prompt) else output
    completion = completion.replace(" equals ", "=").replace(" ", "")  # accept "equals" or "="
    m = re.search(r"Theansweris(-?\d+)because([\d+*/-]+=\d+)", completion)
    if not m:
        return None, None
    return int(m.group(1)), m.group(2)


# load test case
test_path = Path("test_cases_100.json")

with open(test_path, "r", encoding="utf-8") as f:
    test_cases = json.load(f)

# Store evaluation results
results = []
correct_count = 0
category_stats = defaultdict(lambda: {"correct": 0, "total": 0})

gpt.eval()

vocab_size = gpt.config.vocab_size

with torch.no_grad():
    for case in tqdm(test_cases, desc="Evaluating model"):
        prompt = case["question"]
        expected = int(case["answer"])
        category = case["type"]

        # Encode question safely (ignoring characters outside the model vocabulary)
        prompt_ids = [stoi[c] for c in prompt if c in stoi and stoi[c] < vocab_size]
        x = torch.tensor(
            prompt_ids, dtype=torch.long
        ).unsqueeze(0).to(device)

        # Generate prediction
        y, hidden_state = gpt.generate(
            x,
            max_new_tokens=50,
            temperature=1.0,
            top_k=1
        )

        output = decode(y[0].tolist())

        predicted, explanation = parse_output(output, prompt)
        expected_expl = expected_explanation(prompt, expected)

        is_correct = (predicted == expected) and (explanation == expected_expl)

        if is_correct:
            correct_count += 1

        category_stats[category]["total"] += 1
        category_stats[category]["correct"] += int(is_correct)

        results.append({
            "question": prompt,
            "expected": expected,
            "predicted": predicted,
            "expected_explanation": expected_expl,
            "predicted_explanation": explanation,
            "category": category,
            "correct": is_correct,
            "output": output
        })


# Overall accuracy
total = len(test_cases)
accuracy = correct_count / total * 100
print(f"\nOverall Accuracy: {accuracy:.2f}%")

Evaluating model: 100%|██████████| 100/100 [00:15<00:00,  6.40it/s]


Overall Accuracy: 100.00%
